In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, glob, json, warnings, urllib.request
import numpy as np
import pandas as pd
import joblib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve,
)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

RUNID = 'ensemble_run_001'
DRIVEROOT = '/content/drive/MyDrive/tsad_ensemble_runs'
NOTEBOOKTAG = 'conv_ae'

RUNDIR = os.path.join(DRIVEROOT, RUNID, NOTEBOOKTAG)
ARTIFACTDIR = os.path.join(RUNDIR, 'artifacts')
PREDICTIONSDIR = os.path.join(RUNDIR, 'predictions')
CACHEDIR = os.path.join(DRIVEROOT, '_cache')

for d in [ARTIFACTDIR, PREDICTIONSDIR, CACHEDIR]:
    os.makedirs(d, exist_ok=True)

print('RUNDIR       :', RUNDIR)
print('ARTIFACTDIR  :', ARTIFACTDIR)
print('PREDICTIONSDIR:', PREDICTIONSDIR)
print('TF version   :', tf.__version__)


Mounted at /content/drive
RUNDIR       : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/conv_ae
ARTIFACTDIR  : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/conv_ae/artifacts
PREDICTIONSDIR: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/conv_ae/predictions
TF version   : 2.19.0


In [ ]:
# Dataset Paths

MYDRIVE = '/content/drive/MyDrive'
CREDITCARDPATH = os.path.join(MYDRIVE, 'creditcard.csv')

NAB_CANDIDATES = [
    os.path.join(MYDRIVE, 'NAB Dataset'),
    os.path.join(MYDRIVE, 'NAB'),
    os.path.join(MYDRIVE, 'datasets', 'NAB Dataset'),
    os.path.join(MYDRIVE, 'datasets', 'NAB'),
]

def resolve_nab_root(candidates):
    for c in candidates:
        if os.path.isdir(c):
            csvs = glob.glob(os.path.join(c, '**', '*.csv'), recursive=True)
            if len(csvs) > 0:
                return c
    return None

NABROOT = resolve_nab_root(NAB_CANDIDATES)

NABLABELSLOCAL = os.path.join(CACHEDIR, 'nab_combined_windows.json')
NABLABELSURL = 'https://raw.githubusercontent.com/numenta/NAB/master/labels/combined_windows.json'
if not os.path.exists(NABLABELSLOCAL):
    urllib.request.urlretrieve(NABLABELSURL, NABLABELSLOCAL)
with open(NABLABELSLOCAL) as f:
    NABWINDOWSMAP = json.load(f)

assert os.path.exists(CREDITCARDPATH), f'creditcard.csv not found at {CREDITCARDPATH}'
assert NABROOT is not None, 'NAB root not found.'

nab_csvs = [f for f in glob.glob(os.path.join(NABROOT, '**', '*.csv'), recursive=True) if 'README' not in f]
print(f'Credit Card : {CREDITCARDPATH}')
print(f'NAB root    : {NABROOT} ({len(nab_csvs)} CSVs)')
print(f'NAB labels  : {len(NABWINDOWSMAP)} entries')


Credit Card : /content/drive/MyDrive/creditcard.csv
NAB root    : /content/drive/MyDrive/NAB Dataset (58 CSVs)
NAB labels  : 58 entries


In [ ]:
# Shared Utilities

def keep_runs(y, min_len=3):
    y = np.asarray(y, dtype=np.int8).copy()
    n = len(y)
    i = 0
    while i < n:
        if y[i] == 1:
            j = i
            while j < n and y[j] == 1:
                j += 1
            if (j - i) < min_len:
                y[i:j] = 0
            i = j
        else:
            i += 1
    return y

def point_adjust(y_true, y_pred):
    yt = np.asarray(y_true, dtype=np.int8)
    yp = np.asarray(y_pred, dtype=np.int8).copy()
    n = len(yt)
    i = 0
    while i < n:
        if yt[i] == 1:
            j = i
            while j < n and yt[j] == 1:
                j += 1
            if yp[i:j].any():
                yp[i:j] = 1
            i = j
        else:
            i += 1
    return yp

def best_f1_threshold(y, scores, ngrid=300, qlo=0.50, qhi=0.999, min_run=0):
    y = np.asarray(y, dtype=int)
    scores = np.asarray(scores, dtype=float)
    if y.sum() == 0:
        return float(np.percentile(scores, 99.5)), 0.0
    qs = np.linspace(qlo, qhi, ngrid)
    thr_list = np.unique(np.quantile(scores, qs))
    best_t, best_f = float(thr_list[-1]), -1.0
    for t in thr_list:
        p = (scores >= t).astype(int)
        if min_run > 0:
            p = keep_runs(p, min_len=min_run)
        f = f1_score(y, p, zero_division=0)
        if f > best_f:
            best_f, best_t = float(f), float(t)
    return best_t, best_f

def compute_metrics(y, pred, scores=None, prefix=''):
    m = {
        f'{prefix}precision': float(precision_score(y, pred, zero_division=0)),
        f'{prefix}recall': float(recall_score(y, pred, zero_division=0)),
        f'{prefix}f1': float(f1_score(y, pred, zero_division=0)),
    }
    if scores is not None and len(np.unique(y)) == 2:
        m[f'{prefix}rocauc'] = float(roc_auc_score(y, scores))
        m[f'{prefix}prauc'] = float(average_precision_score(y, scores))
    else:
        m[f'{prefix}rocauc'] = float('nan')
        m[f'{prefix}prauc'] = float('nan')
    return m

print('Utilities ready.')


Utilities ready.


In [ ]:
# Credit Card Deep Autoencoder
print('Loading Credit Card data...')
df = pd.read_csv(CREDITCARDPATH).dropna().reset_index(drop=True)
y_all = df['Class'].astype(int).values

X_df = df.copy()
X_df['hoursin'] = np.sin(2 * np.pi * X_df['Time'] / 86400.0)
X_df['hourcos'] = np.cos(2 * np.pi * X_df['Time'] / 86400.0)
X_df['Amountlog'] = np.log1p(X_df['Amount'])
X_df['AmountSq'] = X_df['Amount'] ** 2
for v in ['V1', 'V3', 'V4', 'V7', 'V10', 'V12', 'V14', 'V17']:
    X_df[f'{v}_abs'] = X_df[v].abs()

feature_cols = (
    [f'V{i}' for i in range(1, 29)]
    + ['Amountlog', 'AmountSq', 'hoursin', 'hourcos']
    + [f'{v}_abs' for v in ['V1', 'V3', 'V4', 'V7', 'V10', 'V12', 'V14', 'V17']]
)
X_all = X_df[feature_cols].values.astype(np.float64)

idx = np.arange(len(y_all), dtype=np.int64)
idx_tune, idx_test = train_test_split(idx, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE)
idx_train, idx_val = train_test_split(idx_tune, test_size=0.25, stratify=y_all[idx_tune], random_state=RANDOM_STATE)

X_train_normal = X_all[idx_train][y_all[idx_train] == 0]
X_val = X_all[idx_val]
y_val = y_all[idx_val]
X_test = X_all[idx_test]
y_test = y_all[idx_test]

cc_scaler = RobustScaler()
X_train_normal_sc = np.clip(cc_scaler.fit_transform(X_train_normal), -5, 5)
X_val_sc = np.clip(cc_scaler.transform(X_val), -5, 5)
X_test_sc = np.clip(cc_scaler.transform(X_test), -5, 5)

print(f'Train normal: {len(X_train_normal):,} rows')
print(f'Validation  : {len(X_val):,} rows ({y_val.sum()} frauds)')
print(f'Test        : {len(X_test):,} rows ({y_test.sum()} frauds)')

input_dim = X_train_normal_sc.shape[1]
print(f'Input dim: {input_dim}')

def build_deep_ae(input_dim, bottleneck=6):
    inp = keras.Input(shape=(input_dim,))
    x = layers.Dense(128, activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(16, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    encoded = layers.Dense(bottleneck, activation='relu', name='bottleneck')(x)
    x = layers.Dense(16, activation='relu')(encoded)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    decoded = layers.Dense(input_dim, activation='linear')(x)
    return keras.Model(inp, decoded, name='deep_ae')

ae_model = build_deep_ae(input_dim, bottleneck=6)
ae_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mse')
ae_model.summary()

print('\nTraining Deep AE on normal data only...')
ae_history = ae_model.fit(
    X_train_normal_sc, X_train_normal_sc,
    epochs=150, batch_size=512, validation_split=0.1,
    callbacks=[
        callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor='val_loss'),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=7, min_lr=1e-6),
    ],
    verbose=1,
)
print(f'Deep AE training complete. Best val loss: {min(ae_history.history["val_loss"]):.6f}')


Loading Credit Card data...
Train normal: 170,588 rows
Validation  : 56,962 rows (99 frauds)
Test        : 56,962 rows (98 frauds)
Input dim: 40


Model: "deep_ae"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         5,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck (Dense)              │ (None, 6)              │           102 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 34,382 (134.30 KB)

 Trainable params: 33,422 (130.55 KB)

 Non-trainable params: 960 (3.75 KB)


Training Deep AE on normal data only...
Epoch 1/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 23s 35ms/step - loss: 0.9448 - val_loss: 0.6793 - learning_rate: 0.0010
Epoch 2/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.6947 - val_loss: 0.5904 - learning_rate: 0.0010
Epoch 3/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.6411 - val_loss: 0.5437 - learning_rate: 0.0010
Epoch 4/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.6014 - val_loss: 0.4965 - learning_rate: 0.0010
Epoch 5/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.5681 - val_loss: 0.4719 - learning_rate: 0.0010
Epoch 6/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.5470 - val_loss: 0.4549 - learning_rate: 0.0010
Epoch 7/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.5311 - val_loss: 0.4396 - learning_rate: 0.0010
Epoch 8/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.5197 - val_loss: 0.4272 - learning_rate: 0.0010
Epoch 9/150
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.5096 -

In [ ]:
# Cell 4 — CC Deep AE Scoring (multi-signal)

def ae_anomaly_score(ae_model, X_sc):
    recon = ae_model.predict(X_sc, batch_size=4096, verbose=0)
    errors = X_sc - recon
    mse = np.mean(errors ** 2, axis=1)
    mae = np.mean(np.abs(errors), axis=1)
    max_err = np.max(np.abs(errors), axis=1)
    combined = 0.50 * mse + 0.30 * mae + 0.20 * max_err
    return combined.astype(np.float32)

scores_val = ae_anomaly_score(ae_model, X_val_sc)
scores_test = ae_anomaly_score(ae_model, X_test_sc)

prec, rec, thr = precision_recall_curve(y_val, scores_val)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
if len(thr) > 0:
    best_idx = int(np.nanargmax(f1s))
    cc_threshold = float(thr[best_idx])
    print(f'PR-curve threshold: {cc_threshold:.6f} (val F1={f1s[best_idx]:.4f})')
else:
    cc_threshold = float(np.percentile(scores_val, 99.5))
    print(f'Fallback threshold: {cc_threshold:.6f}')

cc_preds = (scores_test >= cc_threshold).astype(np.int8)
cc_ytrue = y_test.astype(np.int8)

strict = compute_metrics(cc_ytrue, cc_preds, scores_test)

cc_results = {
    'dataset': 'creditcard',
    'protocol': 'semi-supervised strict holdout (20% test)',
    **strict,
    'threshold': cc_threshold,
    'n_anomalies_true': int(cc_ytrue.sum()),
    'n_anomalies_pred': int(cc_preds.sum()),
}

print()
print('=' * 60)
print('CREDIT CARD DEEP AE RESULTS')
print('=' * 60)
for k, v in cc_results.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')

# DEBUG — Deep AE diagnostics
print()
print('=== DEEP AE DIAGNOSTICS ===')
print(f'Normal test MSE mean: {np.mean((X_test_sc[y_test==0] - ae_model.predict(X_test_sc[y_test==0], verbose=0))**2, axis=1).mean():.6f}')
print(f'Fraud test MSE mean:  {np.mean((X_test_sc[y_test==1] - ae_model.predict(X_test_sc[y_test==1], verbose=0))**2, axis=1).mean():.6f}')
fp = int(((cc_preds == 1) & (cc_ytrue == 0)).sum())
fn = int(((cc_preds == 0) & (cc_ytrue == 1)).sum())
tp = int(((cc_preds == 1) & (cc_ytrue == 1)).sum())
print(f'Confusion: TP={tp} FP={fp} FN={fn}')


PR-curve threshold: 3.349371 (val F1=0.6667)

CREDIT CARD DEEP AE RESULTS
  dataset                  : creditcard
  protocol                 : semi-supervised strict holdout (20% test)
  precision                : 0.718447
  recall                   : 0.755102
  f1                       : 0.736318
  rocauc                   : 0.954514
  prauc                    : 0.687856
  threshold                : 3.349371
  n_anomalies_true         : 98
  n_anomalies_pred         : 103

=== DEEP AE DIAGNOSTICS ===
Normal test MSE mean: 0.248405
Fraud test MSE mean:  5.551826
Confusion: TP=74 FP=29 FN=24


In [ ]:
# Export

exported = {}
summaryrows = []

cc_bundle_export = {
    'dataset': 'creditcard',
    'model': 'conv_ae_deep',
    'protocol': 'semi-supervised strict holdout (20% test)',
    'entities': {
        'creditcard': {
            'entityid': 'creditcard',
            'scoresfull': scores_test.astype(np.float32),
            'yfull': cc_ytrue,
            'predfull': cc_preds,
            'rowid': np.arange(len(cc_ytrue), dtype=np.int64),
            'originalrowid': idx_test.astype(np.int64),
            'threshold': cc_threshold,
        }
    }
}
cc_path = os.path.join(PREDICTIONSDIR, 'conv_ae_creditcard_strict.joblib')
joblib.dump(cc_bundle_export, cc_path)
exported['creditcard'] = cc_path
summaryrows.append({
    'dataset': 'creditcard',
    **{k: v for k, v in cc_results.items() if k != 'dataset'},
})
print('Saved', cc_path)

summary = pd.DataFrame(summaryrows)
summary_path = os.path.join(PREDICTIONSDIR, 'conv_ae_summary.csv')
summary.to_csv(summary_path, index=False)
print('Saved', summary_path)
display(summary)

manifest = {
    'runid': RUNID, 'driveroot': DRIVEROOT, 'notebooktag': NOTEBOOKTAG,
    'modelfamily': 'conv_ae', 'exportprotocol': 'ensembleexportv2',
    'artifactsdir': ARTIFACTDIR, 'predictionsdir': PREDICTIONSDIR,
    'exports': exported, 'summarycsv': summary_path,
    'creditcard_originalrowid_included': True,
    'datasets': ['creditcard'],
    'replaces': 'lstm_ae',
    'components': {'creditcard': 'deep_autoencoder'},
}
manifest_path = os.path.join(PREDICTIONSDIR, 'conv_ae_manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print('Saved', manifest_path)

print()
print('Coordinator-ready files in:', PREDICTIONSDIR)
print('Done.')

Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/conv_ae/predictions/conv_ae_creditcard_strict.joblib
Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/conv_ae/predictions/conv_ae_summary.csv


,dataset,protocol,precision,recall,f1,rocauc,prauc,threshold,n_anomalies_true,n_anomalies_pred
0,creditcard,semi-supervised strict holdout (20% test),0.718447,0.755102,0.736318,0.954514,0.687856,3.349371,98,103


Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/conv_ae/predictions/conv_ae_manifest.json

Coordinator-ready files in: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/conv_ae/predictions
Done.
